#### OPTIMIZATION IN DELTA LAKE:

In [0]:
df = spark.read.table("deltalakesushant.default.cloned_table")
display(df)

In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakesushant/default/deltalakevol/optimize/")

## Ran 3 times, and hence 3 part files were created, please note we are specially using 'append' mode, so that we can apply Optimize command on top of it.

In [0]:
%sql
OPTIMIZE DELTA.`/Volumes/deltalakesushant/default/deltalakevol/optimize/`
-- 4th part file is created which have all the data that is in all other 3 files
-- OPTIMIZE command solves the issue of small files, where it just combine all the small files in some big files and it helps in query performance.

In [0]:
%sql
DESCRIBE HISTORY DELTA.`/Volumes/deltalakesushant/default/deltalakevol/optimize/`

![image_1786520195420.png](./image_1786520195420.png "image_1786520195420.png")

#### Z-ORDERING
###### Z-Ordering is a technique to colocate related information in the same set of files. This co-locality is automatically used by Delta Lake in data-skipping algorithms. This behavior dramatically reduces the amount of data that Delta Lake on Apache Spark needs to read. To Z-Order data, you specify the columns to order on in the ZORDER BY clause.

In [0]:
%sql
SELECT * FROM DELTA.`/Volumes/deltalakesushant/default/deltalakevol/optimize/`
WHERE id = 1

In [0]:
%sql
OPTIMIZE DELTA.`/Volumes/deltalakesushant/default/deltalakevol/optimize/` ZORDER BY (id)

In [0]:
%sql
SELECT * FROM DELTA.`/Volumes/deltalakesushant/default/deltalakevol/optimize/`
WHERE id = 1

### LIQUID CLUSTERING
###### Databricks liquid clustering is a flexible data layout optimization feature in Delta Lake that replaces traditional table partitioning and ZORDER. It automatically organizes data based on specified clustering keys, handles incremental updates efficiently, and allows you to change clustering columns over time without rewriting historical files.

In [0]:
%sql
ALTER TABLE deltalakesushant.default.cloned_table
CLUSTER BY (id)

In [0]:
%sql
SELECT * FROM deltalakesushant.default.cloned_table
WHERE id = 1

In [0]:
%sql
-- IN CASE you are not confirm about which column to use for clustering
ALTER TABLE deltalakesushant.default.first_table
CLUSTER BY AUTO